In [2]:
db = [
    {'id': '1', 'question': '發生車禍後，應該立即採取哪些步驟？', 'answer': '確保安全，報警，交換資訊，獲取目擊證人信息。相關法條：《民法》184條，有關賠償的規定。'},
    {'id': '2', 'question': '在家庭暴力情況下，我該如何保護自己？', 'answer': '尋求安全，聯繫當地警察或家暴協助熱線，申請法院保護令。相關法條：《刑法》286-2條，涉及家庭暴力罪。'},
    {'id': '3', 'question': '離婚後，子女的監護權如何決定？', 'answer': '法院考慮子女最佳利益，監護權授予最適合提供良好環境的父母。相關法條：《民法》1059條，有關監護權的規定。'},
    {'id': '4', 'question': '發現公司違法行為，我該如何舉報？', 'answer': '可以匿名舉報給當地稽查機關，也可以聘請律師提起公益訴訟。相關法條：《刑法》234條，有關刑責的規定。'},
    {'id': '5', 'question': '租賃合同終止後，租客應該支付哪些費用？', 'answer': '租客應支付應付的租金和可能的損害賠償。相關法條：《民法》537條，有關租賃合同的規定。'},
    {'id': '6', 'question': '被解雇後有哪些勞動權益保護？', 'answer': '被解雇的員工有權獲得相應的賠償，享有再就業援助。相關法條：《勞動基準法》16條，有關勞動權益的規定。'},
    {'id': '7', 'question': '遺囑如何撰寫以確保合法有效？', 'answer': '遺囑應由成年人親筆簽名，並有足夠的證人。相關法條：《民法》1048條，有關遺囑的規定。'},
    {'id': '8', 'question': '發現鄰居違法建築，應該如何處理？', 'answer': '可以向當地城市規劃或建築管理機構舉報，由相關部門進行調查。相關法條：《都市計劃法》31條，有關城市建設的規定。'},
    {'id': '9', 'question': '在線消費時，如何保護個人隱私？', 'answer': '消費者有權要求商家保護個人隱私，商家應提供隱私政策。相關法條：《個人資料保護法》。'},
    {'id': '10', 'question': '發現商家售賣偽劣商品，應該如何投訴？', 'answer': '可以向當地工商行政管理機構投訴，也可以聘請律師提起侵權訴訟。相關法條：《反不正當競爭法》。'},
]

In [80]:
a = {}
print(f'first: {a}')
a['id']=8
a['animal']='pig'
a['animal']='bird'
print(f'second: {a}')

first: {}
second: {'id': 8, 'animal': 'bird'}


In [6]:
to_be_upsert = []

for every in db:
    temp = {}
    temp['id'] = every['id']
    temp['values'] = every['question']
    # temp['metadata'] = every['question', 'answer']

    to_be_upsert.append(temp)

to_be_upsert


[{'id': '1', 'values': '發生車禍後，應該立即採取哪些步驟？'},
 {'id': '2', 'values': '在家庭暴力情況下，我該如何保護自己？'},
 {'id': '3', 'values': '離婚後，子女的監護權如何決定？'},
 {'id': '4', 'values': '發現公司違法行為，我該如何舉報？'},
 {'id': '5', 'values': '租賃合同終止後，租客應該支付哪些費用？'},
 {'id': '6', 'values': '被解雇後有哪些勞動權益保護？'},
 {'id': '7', 'values': '遺囑如何撰寫以確保合法有效？'},
 {'id': '8', 'values': '發現鄰居違法建築，應該如何處理？'},
 {'id': '9', 'values': '在線消費時，如何保護個人隱私？'},
 {'id': '10', 'values': '發現商家售賣偽劣商品，應該如何投訴？'}]

[
        
    {
        'id': '1',
        'values': get_embedding('發生車禍後，應該立即採取哪些步驟？'),
        'metadata':{'question': '發生車禍後，應該立即採取哪些步驟？', 'answer': '確保安全，報警，交換資訊，獲取目擊證人信息。相關法條：《民法》184條，有關賠償的規定。'}
    },

In [3]:
import pinecone
import openai
import configparser

# Setup
config = configparser.ConfigParser()
config.read('config.ini')

# 調用embedding的OpenAPI
def get_embedding(question):
    openai.api_key = config.get("OpenAI", "api_key")
    response = openai.Embedding.create(
    model="text-embedding-ada-002",
    input = question
    )
    # 提取生成文本中的嵌入向量
    embedding = response['data'][0]['embedding']
    
    return embedding

def init_pinecone(index_name):
    pinecone.init(
        api_key = config.get("pinecone", "api_key"),
        environment='gcp-starter'
    )
    index = pinecone.Index(index_name)
    return index

def make_dataset():
    to_be_upsert = []

    for every in db:
        temp = {}
        temp['id'] = every['id']
        temp['values'] = get_embedding(every['question'])
        temp['metadata'] = {'question': every['question'],'answer': every['answer']}

        to_be_upsert.append(temp)

    return to_be_upsert

def search_from_pinecone(index, query_embedding, k=1):
    results = index.query(vector=query_embedding,
                          top_k=k, include_metadata=True, namespace='first_try')
    return results

# #################### Main function ####################
index = init_pinecone('test01') 
index.upsert(
    vectors=make_dataset(),
    namespace='first_try'
)


{'upserted_count': 10}

In [4]:
# 進行相似性搜索
try:
    index = init_pinecone("test01")
    question = "發生車禍怎麼辦？"
    query_embedding = get_embedding(question)
    qa_results = search_from_pinecone(index, query_embedding, k=5)   # k為尋找相似性最高
    # print(f"qa_results : {qa_results}")
    
    # a_result =[]
    
    # for result in qa_results['matches']:
    #     a_result.append(result['metadata']['answer']) 
    
    # print(a_result)
    #print(qa_results)
    print(qa_results["matches"])
    #print(qa_results["matches"][0]["metadata"]["answer"])

except Exception as e:
    print(f"An error occurred: {e}")

[{'id': '1',
 'metadata': {'answer': '確保安全，報警，交換資訊，獲取目擊證人信息。相關法條：《民法》184條，有關賠償的規定。',
              'question': '發生車禍後，應該立即採取哪些步驟？'},
 'score': 0.931464076,
 'values': []}, {'id': '2',
 'metadata': {'answer': '尋求安全，聯繫當地警察或家暴協助熱線，申請法院保護令。相關法條：《刑法》286-2條，涉及家庭暴力罪。',
              'question': '在家庭暴力情況下，我該如何保護自己？'},
 'score': 0.849883795,
 'values': []}, {'id': '8',
 'metadata': {'answer': '可以向當地城市規劃或建築管理機構舉報，由相關部門進行調查。相關法條：《都市計劃法》31條，有關城市建設的規定。',
              'question': '發現鄰居違法建築，應該如何處理？'},
 'score': 0.84224546,
 'values': []}, {'id': '4',
 'metadata': {'answer': '可以匿名舉報給當地稽查機關，也可以聘請律師提起公益訴訟。相關法條：《刑法》234條，有關刑責的規定。',
              'question': '發現公司違法行為，我該如何舉報？'},
 'score': 0.838839829,
 'values': []}, {'id': '3',
 'metadata': {'answer': '法院考慮子女最佳利益，監護權授予最適合提供良好環境的父母。相關法條：《民法》1059條，有關監護權的規定。',
              'question': '離婚後，子女的監護權如何決定？'},
 'score': 0.82423532,
 'values': []}]


###################################################################################################

In [5]:
import pinecone
import openai
import configparser

# Setup
config = configparser.ConfigParser()
config.read('config.ini')

# 調用embedding的OpenAPI
def get_embedding(question):
    openai.api_key = config.get("OpenAI", "api_key")
    response = openai.Embedding.create(
    model="text-embedding-ada-002",
    input = question
    )
    # 提取生成文本中的嵌入向量
    embedding = response['data'][0]['embedding']
    
    return embedding

def init_pinecone(index_name):
    pinecone.init(
        api_key = config.get("pinecone", "api_key"),
        environment='gcp-starter'
    )
    index = pinecone.Index(index_name)
    return index

def make_dataset():
    to_be_upsert = []

    for every in db:
        temp = {}
        temp['id'] = every['id']
        temp['values'] = get_embedding(every['question'])
        temp['metadata'] = {'question': every['question'],'answer': every['answer']}

        to_be_upsert.append(temp)

    return to_be_upsert

def search_from_pinecone(index, query_embedding, k=1):
    results = index.query(vector=query_embedding,
                          top_k=k, include_metadata=True, namespace='first_try')
    return results

#######################################################################################
# Input: question
# Output: If in db, then return [{'qustion':question, 'answer':answer, 'score':score}, {},{}]. If in gpt, then return 'gpt'
# Ouptput_limit: score >= 0.8
def fetch_db_or_gpt(question: str):
    index = init_pinecone("test01")
    query_embedding = get_embedding(question)
    qa_results = search_from_pinecone(index, query_embedding, k=10)
    
    similarity_matches = []
    
    for every_info in qa_results['matches']:
        # If score >= 0.8
        if every_info['score'] >= 0.8:
            temp={}
            temp['question']=every_info['metadata']['question']
            temp['answer']=every_info['metadata']['answer']
            temp['score']=every_info['score']

            similarity_matches.append(temp)

    if similarity_matches==[]:
        return 'gpt'
    else:
        return similarity_matches
    

In [6]:
fetch_db_or_gpt('車禍')

[{'question': '發生車禍後，應該立即採取哪些步驟？',
  'answer': '確保安全，報警，交換資訊，獲取目擊證人信息。相關法條：《民法》184條，有關賠償的規定。',
  'score': 0.841214538}]

In [121]:
def find_score(score):
    if qa_results['matches'][0]['score'] >= 0.85:
        return qa_results['matches']
    else:
        return 'not find'

In [122]:
find_score('車禍')

[{'id': '1',
  'metadata': {'answer': '確保安全，報警，交換資訊，獲取目擊證人信息。相關法條：《民法》184條，有關賠償的規定。',
               'question': '發生車禍後，應該立即採取哪些步驟？'},
  'score': 0.931464076,
  'values': []},
 {'id': '2',
  'metadata': {'answer': '尋求安全，聯繫當地警察或家暴協助熱線，申請法院保護令。相關法條：《刑法》286-2條，涉及家庭暴力罪。',
               'question': '在家庭暴力情況下，我該如何保護自己？'},
  'score': 0.849883795,
  'values': []},
 {'id': '8',
  'metadata': {'answer': '可以向當地城市規劃或建築管理機構舉報，由相關部門進行調查。相關法條：《都市計劃法》31條，有關城市建設的規定。',
               'question': '發現鄰居違法建築，應該如何處理？'},
  'score': 0.84224546,
  'values': []},
 {'id': '4',
  'metadata': {'answer': '可以匿名舉報給當地稽查機關，也可以聘請律師提起公益訴訟。相關法條：《刑法》234條，有關刑責的規定。',
               'question': '發現公司違法行為，我該如何舉報？'},
  'score': 0.838839829,
  'values': []},
 {'id': '3',
  'metadata': {'answer': '法院考慮子女最佳利益，監護權授予最適合提供良好環境的父母。相關法條：《民法》1059條，有關監護權的規定。',
               'question': '離婚後，子女的監護權如何決定？'},
  'score': 0.82423532,
  'values': []}]

In [70]:
import pandas as pd

df = pd.read_csv('高點法律題庫-5.xlsx - 訓練集.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   題號      129 non-null    object
 1   題目      129 non-null    object
 2   答案      129 non-null    object
 3   法律關鍵字   129 non-null    object
dtypes: object(4)
memory usage: 4.2+ KB


In [ ]:
from pprint import pprint

def extract_from_long_string(long_string):
    for index, every_word in enumerate(long_string):
        if long_string[index:index+4]=='命題意旨':
            question = long_string[:index]

    pprint(question)

for key, value in d['題目,答案,參考法條,法律關鍵字'].items():
    extract_from_long_string(value)
    break

In [44]:
def extract_from_long_string(long_string):
    
    question = ""
    answer = ""
    reference = ""
    keyword = ""

    catch_start_index = 0

    for index, _ in enumerate(long_string):

        if long_string[index:index+4]=='命題意旨':
            question = long_string[catch_start_index:index]
            catch_start_index = index

        if long_string[index:index+3]=='(一)':
            reference = long_string[catch_start_index:index]
            catch_start_index = index
        
        if long_string[index:index+8]=='民法與民事訴訟法':
            answer = long_string[catch_start_index:index]
            catch_start_index = index

            keyword = long_string[catch_start_index:]


    return question, answer, reference, keyword





new_list = []

for key, value in d['題目,答案,參考法條,法律關鍵字'].items():
    question, answer, reference, keyword = extract_from_long_string(value)

    temp = {}
    temp['題目'] = question
    temp['答案'] = answer
    temp['參考法條'] = reference
    temp['法律關鍵字'] = keyword
    new_list.append(temp)

blank = []
for index,i in enumerate(new_list):
    if i['題目']=='':
        blank.append(index)

In [58]:
for i in range(len(d['題目,答案,參考法條,法律關鍵字'])):
    question_seperate = ''

    sentence = d['題目,答案,參考法條,法律關鍵字'][i]
    for index, word in enumerate(sentence):
        if word=='？':
            question_seperate = sentence[index:index+6]
    print(question_seperate)

？ 命題意旨
？命題意旨 
？ 命題意旨
？對此法無明
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 意旨 遺
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 命題意旨
？,甲得向乙
？第二個小題
？ 命題意旨
？ 命題意旨
？,(一) 
？命題意旨 
？ 命題意旨
？ 命題意旨
？命題意旨 
？ 命題意旨
？ 命題意旨
？對此法無明
？ 命題意旨
？ 命題意旨
？ 決議：承
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 決議：承
？ 命題意旨
？,(一)本
？,(一)甲
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 命題意旨
？按「占有物
？此涉及法條
？實務見解認
？ 命題意旨
？）並據此要
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 頁 3-
？ 審查意見
？參高等法院
？ 命題意旨
？ 1 最高
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 1 最高
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 1 最高
？ 命題意旨
？ 命題意旨
？對此，有論
？,(一)甲
？ 命題意旨
？ 命題意旨
？ 命題意旨
？ 命題意旨
？對此，可能
？  2 
？ 命題意旨
？ 命題意旨
？,(一)就
？,本題涉及
？",(一)
？ (二)丙
？ 命題意旨
？ 命題意旨
？ 命題意旨
？此涉及第 
？ 命題意旨
？ 命題意旨
？ 本案甲之
？就本案丁之
？ 1.按民
？ (1)按
？ 命題意旨
？ 命題意旨
？本文以為本
？分述如下：
？命題意旨 
？是否得依借
？命題意旨 
？分述如下：
？命題意旨 
？是否得依借
？命題意旨 
？不可一概而
？端視乙之行
？ 《民事訴
？近期實務見
？ 命題意旨
？ (A) 
？ (1)或
？ 答題關鍵
？ 答題關鍵
？又因兩造間
？ 答題關鍵
？對此，學者
？ 答題關鍵
？ 答題關鍵
？ 滅。 推
？說明如下：
？ 3.《月

？對此，實務
？ 答題關鍵
？學說上向有
？,(一)乙
？ (A)戊
？答題關鍵 
？答題關鍵本
？ 答題關鍵
？最高法院則
？ 第一小題
？ 第一小題
？非無疑問。
？(25 分
？(25 分
？(25 分
？(25 分
？ 命題意旨
？ (A)乙
？,法院應判
？惟查，賦予
？就此，雖實
？亦非無研求
？本件 B地


？應由當事人
？就此，民法
？ 就前揭不
？就此，實務
？依通說見解
？  2 
？ 命